# Independent assembly comparison

This notebook compares four **already generated** H5AD assemblies (`mean`, `random`, `within_cluster`, and `outside_cluster`). It never reconstructs or writes an H5AD. For each requested broad type, it keeps exact real-ID and gene-name intersections, rebuilds a normalized/log1p working copy, and runs PCA, neighbors, and Leiden independently for every method.

The original spatial H5AD supplies subtype labels and coordinates only after clustering. Missing types, IDs, genes, and labels stay visible; they are not replaced by zero. ARI/NMI and contingency tables describe each requested resolution without choosing a best resolution or a winning method.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from revise.analysis.assembly_comparison import (
    INPUT_ASSUMPTIONS,
    assert_input_hashes_unchanged,
    compare_assembly_methods,
    load_assembly_inputs,
    plot_spatial_comparison,
)

PARAMETERS = {
    'method_paths': {
        'mean': 'output/assembly/mean.h5ad',
        'random': 'output/assembly/random.h5ad',
        'within_cluster': 'output/assembly/within_cluster.h5ad',
        'outside_cluster': 'output/assembly/outside_cluster.h5ad',
    },
    'baseline_path': 'output/assembly/original_spatial.h5ad',
    'type_aliases': {
        'T': ['T', 'T cell', 'T cells', 'T-cell'],
        'Macro': ['Macro', 'Macrophage', 'Macrophages'],
        'CAF': ['CAF', 'CAFs', 'Fibroblast', 'Fibroblasts'],
    },
    'broad_column': 'revise_Level1',
    'broad_fallback': 'revise_Level1',
    'baseline_subtype_column': 'SVC_cluster',
    'spatial_key': 'spatial',
    'seed': 42,
    'resolutions': [0.6, 0.7, 0.8],
    'plot_resolution': 0.7,
}
config_path = os.environ.get('REVISE_ASSEMBLY_COMPARISON_CONFIG')
if config_path:
    PARAMETERS.update(json.loads(Path(config_path).read_text(encoding='utf-8')))
PARAMETERS

## 1. Declare the comparison boundary

The generated matrices are declared by the caller to be finite, nonnegative, unlogged linear expression. Fractional mixtures and other linear-scale values are valid. The helper validates this boundary before making working copies. Exact names define correspondence; row position does not. The broad-column fallback is recorded in the coverage table rather than applied silently.

In [ ]:
assumption_table = pd.Series(INPUT_ASSUMPTIONS, name='declared behavior').to_frame()
display(assumption_table)

## 2. Read native H5ADs and fingerprint them

Hashes are captured before analysis and checked again at the end. Shapes below describe native files, before any type selection or intersection.

In [ ]:
inputs = load_assembly_inputs(
    PARAMETERS['method_paths'],
    PARAMETERS['baseline_path'],
)
input_summary = pd.DataFrame([
    {
        'object': name,
        'path': str(inputs.paths[name]),
        'n_obs': (inputs.baseline if name == 'baseline' else inputs.methods[name]).n_obs,
        'n_vars': (inputs.baseline if name == 'baseline' else inputs.methods[name]).n_vars,
        'sha256': inputs.hashes[name],
    }
    for name in [*inputs.methods, 'baseline']
])
display(input_summary)

## 3. Build equal scopes, then cluster each method independently

For a broad type, every method and the baseline must contain the same real IDs. Every expression method must also contain the same genes. Shared genes are sorted lexicographically for a stable feature order. Each method then receives its own normalization, log1p, PCA, neighbor graph, and Leiden run; baseline labels are not inputs to any of those steps.

In [ ]:
comparison = compare_assembly_methods(
    inputs.methods,
    inputs.baseline,
    type_aliases=PARAMETERS['type_aliases'],
    broad_column=PARAMETERS['broad_column'],
    broad_fallback=PARAMETERS.get('broad_fallback', 'revise_Level1'),
    baseline_subtype_column=PARAMETERS['baseline_subtype_column'],
    resolutions=PARAMETERS['resolutions'],
    seed=PARAMETERS['seed'],
)
display(comparison.coverage)

In [ ]:
scope_summary = pd.DataFrame([
    {
        'broad_type': name,
        'status': result.status,
        'issues': '; '.join(result.issues) or None,
        'shared_id_count': len(result.shared_ids),
        'shared_gene_count': len(result.shared_genes),
        'shared_id_preview': tuple(result.shared_ids[:5]),
        'shared_gene_preview': tuple(result.shared_genes[:10]),
    }
    for name, result in comparison.by_type.items()
])
display(scope_summary)

## 4. Read metrics and label correspondences

ARI and NMI use only exact shared IDs whose baseline subtype label is present. The matched and missing label counts remain beside every score. Contingency tables retain the label-to-cluster structure; cluster numbers are method-local and are not treated as shared identities. No score is promoted to an automatic winner.

In [ ]:
display(comparison.metrics)
for broad_type, result in comparison.by_type.items():
    if result.status != 'ok':
        print(f'{broad_type}: unavailable — {"; ".join(result.issues)}')
        continue
    for method in inputs.methods:
        for resolution in PARAMETERS['resolutions']:
            print(f'{broad_type} | {method} | resolution={resolution}')
            display(result.contingencies[(method, float(resolution))])

## 5. Inspect spatial organization at one declared resolution

Each pair uses original spatial coordinates. The left panel shows the configured original subtype field; the right panel shows the new expression-only Leiden labels for one method. This is a visual comparison, not an extra scoring or selection rule.

In [ ]:
plot_resolution = float(PARAMETERS['plot_resolution'])
for broad_type, result in comparison.by_type.items():
    if result.status != 'ok':
        continue
    for method in inputs.methods:
        figure, _ = plot_spatial_comparison(
            result,
            inputs.baseline,
            method=method,
            resolution=plot_resolution,
            spatial_key=PARAMETERS.get('spatial_key', 'spatial'),
        )
        display(figure)
        plt.close(figure)

## 6. Confirm native inputs stayed unchanged

This final check re-hashes every source H5AD. Any change is an error rather than an analysis output.

In [ ]:
assert_input_hashes_unchanged(inputs)
print('native input hashes unchanged')